# 17. 당뇨병 위험 예측 함수 만들기

이번 장에서는 학습한 모델을 나중에 다시 사용할 수 있도록 저장하고, 새 입력값을 예측하는 함수를 만듭니다.

중요한 점:

```text
모델만 저장하면 부족합니다.
scaler와 feature_names도 함께 저장해야 합니다.
```

## 1. 라이브러리 불러오기

이번 장에서는 `joblib`을 사용합니다.

`joblib`은 scaler 같은 파이썬 객체를 파일로 저장하고 불러올 때 자주 씁니다.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from keras.models import Sequential, load_model
from keras.layers import Input, Dense, Dropout
from keras.callbacks import EarlyStopping

## 2. 데이터 준비

이전 장들과 같은 당뇨병 이진 분류 데이터를 사용합니다.

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\diabetes_or_cardiovascular\diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
TARGET = "Diabetes_binary"

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET])
y = df[TARGET]

# feature_names는 학습 때 사용한 컬럼 순서입니다.
# 나중에 새 입력값을 예측할 때 반드시 이 순서로 맞춰야 합니다.
feature_names = list(X.columns)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("feature 개수:", len(feature_names))
print(feature_names)

## 3. Scaler 학습과 변환

`StandardScaler`는 학습 데이터의 평균과 표준편차를 기억합니다.

나중에 새 입력값도 이 scaler로 변환해야 합니다.

In [ ]:
scaler = StandardScaler()

# fit_transform()은 학습 데이터에서 평균/표준편차를 배우고 변환까지 합니다.
X_train_scaled = scaler.fit_transform(X_train)

# transform()은 이미 배운 기준으로 검증 데이터를 변환합니다.
X_val_scaled = scaler.transform(X_val)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)

## 4. 예측용 MLP 모델 학습

데모 연결이 목적이므로 이전 장과 비슷한 MLP를 사용합니다.

In [ ]:
input_dim = X_train_scaled.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

## 5. 모델, scaler, feature_names 저장

모델은 `.keras`로 저장합니다.

scaler와 feature_names는 `joblib`으로 저장합니다.

In [ ]:
OUTPUT_DIR = Path(r"C:\work\deepLearning\model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUTPUT_DIR / "diabetes_prediction_pipeline_model.keras"
scaler_path = OUTPUT_DIR / "diabetes_prediction_scaler.joblib"
feature_names_path = OUTPUT_DIR / "diabetes_prediction_feature_names.joblib"

# Keras 모델을 저장합니다.
model.save(model_path)

# joblib.dump()는 파이썬 객체를 파일로 저장합니다.
joblib.dump(scaler, scaler_path)
joblib.dump(feature_names, feature_names_path)

print("모델 저장:", model_path)
print("scaler 저장:", scaler_path)
print("feature_names 저장:", feature_names_path)

## 6. 저장한 파일 다시 불러오기

실제 데모 앱에서는 이미 저장된 파일을 불러와서 예측합니다.

그래서 여기서도 저장한 파일을 다시 불러오는 흐름을 연습합니다.

In [ ]:
# load_model()은 저장된 Keras 모델을 다시 불러옵니다.
loaded_model = load_model(model_path)

# joblib.load()는 joblib.dump()로 저장한 객체를 다시 불러옵니다.
loaded_scaler = joblib.load(scaler_path)
loaded_feature_names = joblib.load(feature_names_path)

print("불러온 feature 개수:", len(loaded_feature_names))
print(loaded_feature_names)

## 7. 새 입력 샘플 만들기

새 사람의 건강 지표를 딕셔너리로 만들어 봅니다.

실제 앱에서는 사용자가 화면에서 입력한 값이 이런 딕셔너리 형태로 들어올 수 있습니다.

In [ ]:
# 검증 데이터의 첫 번째 행을 예시 샘플로 사용합니다.
# to_dict()는 한 행을 딕셔너리로 바꿉니다.
sample_input = X_val.iloc[0].to_dict()

print("예시 입력:")
sample_input

## 8. 예측 함수 만들기

이 함수가 이번 장의 핵심입니다.

입력 딕셔너리를 받아서 다음 순서로 처리합니다.

```text
딕셔너리
-> DataFrame 한 행
-> feature_names 순서로 정렬
-> scaler.transform
-> model.predict
-> threshold로 0/1 판단
```

In [ ]:
def predict_diabetes_risk(input_dict, model, scaler, feature_names, threshold=0.5):
    """새 건강 지표 입력값을 받아 당뇨병 있음 클래스의 예측 점수와 0/1 판단을 반환합니다."""
    
    # 딕셔너리를 DataFrame 한 행으로 바꿉니다.
    input_df = pd.DataFrame([input_dict])
    
    # 학습 때 사용한 컬럼이 모두 있는지 확인합니다.
    missing_features = [name for name in feature_names if name not in input_df.columns]
    if missing_features:
        raise ValueError(f"입력값에 누락된 컬럼이 있습니다: {missing_features}")
    
    # 학습 때와 같은 컬럼 순서로 정렬합니다.
    input_df = input_df[feature_names]
    
    # 저장된 scaler로 변환합니다.
    input_scaled = scaler.transform(input_df)
    
    # 모델 예측 확률을 계산합니다.
    probability = float(model.predict(input_scaled, verbose=0)[0][0])
    
    # threshold 기준으로 0/1을 결정합니다.
    predicted_label = int(probability >= threshold)
    
    return {
        "probability": probability,
        "threshold": threshold,
        "predicted_label": predicted_label,
    }

## 9. 예측 함수 사용하기

저장했다가 다시 불러온 모델과 scaler를 사용합니다.

In [ ]:
prediction = predict_diabetes_risk(
    input_dict=sample_input,
    model=loaded_model,
    scaler=loaded_scaler,
    feature_names=loaded_feature_names,
    threshold=0.5
)

prediction

## 10. 예측 결과를 안전한 문장으로 바꾸기

건강 데이터에서는 표현이 중요합니다.

진단처럼 말하지 않고, 교육용 모델의 예측 점수로 설명합니다.

In [ ]:
def format_prediction_message(prediction):
    """예측 결과 딕셔너리를 사용자에게 보여줄 문장으로 바꿉니다."""
    
    probability = prediction["probability"]
    threshold = prediction["threshold"]
    predicted_label = prediction["predicted_label"]
    
    if predicted_label == 1:
        result_text = "당뇨병 있음 클래스의 예측 점수가 threshold 이상입니다."
    else:
        result_text = "당뇨병 있음 클래스의 예측 점수가 threshold 미만입니다."
    
    message = (
        f"예측 점수: {probability:.3f}\n"
        f"사용 threshold: {threshold:.2f}\n"
        f"판단: {result_text}\n\n"
        "주의: 이 결과는 의학적 진단이 아니며, 딥러닝 학습을 위한 교육용 모델 출력입니다."
    )
    
    return message


print(format_prediction_message(prediction))

## 11. Threshold를 바꿔서 예측해보기

같은 입력값이라도 threshold를 바꾸면 최종 0/1 판단이 달라질 수 있습니다.

In [ ]:
for threshold in [0.3, 0.5, 0.7]:
    prediction = predict_diabetes_risk(
        input_dict=sample_input,
        model=loaded_model,
        scaler=loaded_scaler,
        feature_names=loaded_feature_names,
        threshold=threshold
    )
    
    print("=" * 50)
    print(format_prediction_message(prediction))

## 12. 입력값 누락 오류 확인

실제 앱에서는 사용자가 값을 빠뜨릴 수 있습니다.

예측 함수에서 누락 컬럼을 확인하면 문제를 빨리 찾을 수 있습니다.

In [ ]:
# 일부러 BMI 컬럼을 제거한 잘못된 입력을 만듭니다.
bad_input = sample_input.copy()
bad_input.pop("BMI")

# 아래 코드는 오류 확인용입니다.
# 실행하면 ValueError가 발생하며, 누락된 컬럼 이름을 알려줍니다.
# predict_diabetes_risk(bad_input, loaded_model, loaded_scaler, loaded_feature_names)

## 13. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. 표 데이터 모델은 모델 파일만 저장하면 부족하다.
2. scaler와 feature_names를 함께 저장해야 한다.
3. 새 입력값은 학습 때와 같은 컬럼 순서로 정렬해야 한다.
4. 예측 함수는 전처리와 모델 예측을 하나의 흐름으로 묶는다.
5. 건강 예측 결과는 의학적 진단처럼 표현하면 안 된다.
```

## 과제

1. `sample_input`의 BMI 값을 바꿔서 예측 점수가 어떻게 달라지는지 확인해보세요.
2. threshold를 0.3, 0.5, 0.7로 바꿨을 때 판단이 바뀌는지 확인해보세요.
3. feature_names를 저장하지 않으면 어떤 문제가 생길지 설명해보세요.
4. 이 예측 함수를 Streamlit 앱에 연결한다면 어떤 입력 UI가 필요할지 적어보세요.